In [1]:
import os
import pandas as pd
import joblib
import numpy as np
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score, recall_score, f1_score
from sklearn.model_selection import RandomizedSearchCV

In [2]:
# Paths
DATA_DIR = '../data/processed_data/'
MODEL_DIR = '../models/'

os.makedirs(MODEL_DIR, exist_ok=True)

In [3]:
# Load Processed Data
def load_data():
    X_train = np.load(os.path.join(DATA_DIR, 'X_train.npy'))
    X_test = np.load(os.path.join(DATA_DIR, 'X_test.npy'))
    y_train = np.load(os.path.join(DATA_DIR, 'y_train.npy'))
    y_test = np.load(os.path.join(DATA_DIR, 'y_test.npy'))

    print("Processed data loaded successfully.")
    return X_train, X_test, y_train, y_test

In [4]:
X_train, X_test, y_train, y_test = load_data()

Processed data loaded successfully.


In [5]:
model = joblib.load(os.path.join(MODEL_DIR, 'xgb_fraud_model.pkl'))


In [6]:
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

In [7]:
y_proba = model.predict_proba(X_test)[:, 1]


In [8]:
auc = roc_auc_score(y_test, y_proba)
cm = confusion_matrix(y_test, y_pred)
report = classification_report(y_test, y_pred)
pr_auc = average_precision_score(y_test, y_proba)

print("\nClassification Report:\n", report)
print("Confusion Matrix:\n", cm)
print(f"ROC AUC Score: {auc:.4f}")
print(f"Precision-Recall AUC Score: {pr_auc:.4f}")


Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.88      0.83      0.85        98

    accuracy                           1.00     56962
   macro avg       0.94      0.91      0.93     56962
weighted avg       1.00      1.00      1.00     56962

Confusion Matrix:
 [[56853    11]
 [   17    81]]
ROC AUC Score: 0.9731
Precision-Recall AUC Score: 0.8753


In [9]:
fraud_ratio = (y_train == 0).sum() / (y_train == 1).sum()
print(f"Fraud Ratio in Training Data: {fraud_ratio:.2f}")

Fraud Ratio in Training Data: 577.29


In [10]:
xgb = XGBClassifier(
    objective='binary:logistic',
    eval_metric='auc',
    scale_pos_weight=fraud_ratio,
    random_state=42
)

In [11]:
param_dist = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [3, 5, 7, 9],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "gamma": [0, 0.1, 0.3],
    "min_child_weight": [1, 3, 5]
}

In [12]:
random_search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_dist,
    n_iter=30,
    scoring='roc_auc',
    cv=5,
    verbose=2,
    n_jobs=-1,
    random_state=42
)

In [13]:
random_search.fit(X_train, y_train)

best_model = random_search.best_estimator_

print("Best Parameters:", random_search.best_params_)

Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best Parameters: {'subsample': 0.8, 'n_estimators': 500, 'min_child_weight': 5, 'max_depth': 9, 'learning_rate': 0.1, 'gamma': 0.1, 'colsample_bytree': 0.8}


In [14]:
xgb_proba = best_model.predict_proba(X_test)[:, 1]

In [15]:
rf = RandomForestClassifier(
    class_weight="balanced",  # IMPORTANT
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [16]:
param_dist = {
    "n_estimators": [200, 300, 500],
    "max_depth": [None, 10, 20, 30],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"],
    "bootstrap": [True, False]
}

In [17]:
search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=20,
    scoring="roc_auc",  # TRAINING metric
    cv=5,
    verbose=2,
    n_jobs=-1,
    random_state=42
)

search.fit(X_train, y_train)

best_rf = search.best_estimator_

Fitting 5 folds for each of 20 candidates, totalling 100 fits


KeyboardInterrupt: 

In [ ]:
rf_prob  = best_rf.predict_proba(X_test)[:,1]

In [ ]:
threshold = 0.3

# Predictions
xgb_pred = (xgb_proba > threshold).astype(int)
rf_pred  = (rf_prob > threshold).astype(int)

In [ ]:
results = {
    "Model": ["XGBoost", "Random Forest"],

    "ROC-AUC": [
        roc_auc_score(y_test, xgb_proba),
        roc_auc_score(y_test, rf_prob)
    ],

    "PR-AUC": [
        average_precision_score(y_test, xgb_proba),
        average_precision_score(y_test, rf_prob)
    ],

    "Recall": [
        recall_score(y_test, xgb_pred),
        recall_score(y_test, rf_pred)
    ],

    "F1-Score": [
        f1_score(y_test, xgb_pred),
        f1_score(y_test, rf_pred)
    ]
}

In [ ]:
df_results = pd.DataFrame(results)
print(df_results)